# Galerie données mixtes — 03 · Encoder 1 206 produits sans exploser 🟠

> **Étagère optionnelle post-M4** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🟠 **à compléter** — les cellules `# TODO` sont à toi.
> **Pas de solution fournie** ; le guidage décroît au fil du notebook.
> **Durée** : ~1 h 30 · **Pré-requis** : notebook 01 fait.
> **Fiches à garder ouvertes** : `fiche_preprocessing.pdf` (section haute
> cardinalité) · `fiche_pattern_preparation_donnees.md` · le notebook 01
> comme référence de code.

## Le contexte

Léa Fontan revient vers toi :

> « Le merchandising est convaincu que **certains produits concentrent
> l'insatisfaction**. Vos modèles ignorent l'identifiant produit
> (`clothing_id`) et le rayon détaillé (`class_name`) — ajoutez-les.
> Et il y a un cas qui m'inquiète : les **845 avis "note seule", sans
> texte** — pour eux, votre TF-IDF ne sert à rien. »

Problème : `clothing_id` a **1 206 modalités**. Le OneHot que tu connais
produirait 1 206 colonnes pour une seule variable — et chaque nouveau produit
au catalogue serait une modalité inconnue. C'est le problème de la **haute
cardinalité**, et il se traite avec des encodages dédiés que tu vas coder
et brancher toi-même.

## Setup + chargement (donné — détaillé au notebook 01)

Note la ligne `astype(str)` : un identifiant est une **étiquette**, pas une
quantité. Le laisser en numérique inviterait le modèle à calculer des
moyennes d'identifiants — un non-sens silencieux.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42

URL = ("https://raw.githubusercontent.com/AFAgarap/ecommerce-reviews-analysis/"
       "master/Womens%20Clothing%20E-Commerce%20Reviews.csv")

try:
    df = pd.read_csv(URL, index_col=0)
except Exception as err:
    print(f"Téléchargement impossible ({err}) — lecture du CSV local.")
    df = pd.read_csv("data/clothing_reviews.csv", index_col=0)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


def vers_satisfaction(note: int) -> str:
    if note <= 2:
        return "insatisfaite"
    if note == 3:
        return "mitigée"
    return "satisfaite"


df["satisfaction"] = df["rating"].apply(vers_satisfaction)
df = df.drop(columns=["rating", "recommended_ind", "positive_feedback_count"])
df["review_text"] = df["review_text"].fillna("")
df["longueur_avis"] = df["review_text"].str.len()
df["clothing_id"] = df["clothing_id"].astype(str)  # un identifiant n'est PAS un nombre
print(df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

colonnes_num = ["age", "longueur_avis"]
colonnes_cat = ["division_name", "department_name"]
colonne_txt = "review_text"

X = df[["clothing_id", "class_name"] + colonnes_num + colonnes_cat + [colonne_txt]]
y = df["satisfaction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## [1] État des lieux (donné)

Avant d'encoder, on mesure à quoi on a affaire.

In [ ]:
for col in ["clothing_id", "class_name", "division_name", "department_name"]:
    print(f"{col:18s} {X_train[col].nunique():5d} modalités "
          f"| top : {X_train[col].value_counts().head(2).to_dict()}")

`class_name` (20 modalités) tient encore en OneHot. `clothing_id` (plus de
1 100 modalités rien que sur le train), non :

1. **explosion de colonnes** → mémoire, temps, sur-apprentissage sur les
   produits rares ;
2. **modalités inconnues** : chaque nouveau produit du catalogue arriverait
   en vecteur tout-zéros ;
3. le modèle apprendrait par cœur les produits du train au lieu d'apprendre
   des régularités.

## [2] La preuve par l'expérience (donné)

On mesure d'abord — baseline du notebook 01, puis OneHot naïf sur
`clothing_id`. La fonction `evalue` t'accompagne tout le notebook.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def branches_de_base() -> list:
    """Les 3 branches du notebook 01 : num + cat + texte."""
    return [
        ("num", Pipeline([("imputation", SimpleImputer(strategy="median")),
                          ("echelle", StandardScaler())]), colonnes_num),
        ("cat", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), colonnes_cat),
        ("txt", TfidfVectorizer(max_features=5000, min_df=3, stop_words="english"), colonne_txt),
    ]


def evalue(nom: str, branches: list) -> float:
    """Entraîne une régression logistique sur ces branches et affiche f1_macro + largeur."""
    pipe = Pipeline([("preparation", ColumnTransformer(branches)),
                     ("modele", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    pipe.fit(X_train, y_train)
    n_colonnes = pipe.named_steps["preparation"].transform(X_train.head(50)).shape[1]
    f1 = f1_score(y_test, pipe.predict(X_test), average="macro")
    print(f"{nom:48s} f1_macro={f1:.3f}   colonnes={n_colonnes}")
    return f1


evalue("baseline (features du notebook 01)", branches_de_base())
evalue("+ clothing_id en OneHot naïf",
       branches_de_base() + [("cid", OneHotEncoder(handle_unknown="ignore"), ["clothing_id"])])

Constate : **plus de 1 100 colonnes ajoutées pour un f1_macro qui BAISSE.**
Le OneHot naïf sur de la haute cardinalité, c'est payer plus cher pour faire
pire. Maintenant, à toi.

## 🎯 [3] À toi — le frequency encoding (guidage fort)

L'idée la plus simple qui marche : remplacer chaque modalité par sa
**fréquence relative observée sur le train**. Une seule colonne en sortie,
quelle que soit la cardinalité, et une modalité inconnue devient
naturellement `0.0`.

Complète les deux `TODO` de ce transformer scikit-learn maison — la cellule
de test juste en dessous te dit si c'est bon.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Remplace chaque modalité par sa fréquence relative apprise sur le train.

    Sortie : 1 colonne. Modalité jamais vue au fit -> 0.0.
    """

    def fit(self, X, y=None):
        colonne = X.iloc[:, 0]
        # TODO 1 — stocke la fréquence relative de chaque modalité dans
        # self.frequences_  (indice : value_counts avec normalize=True)
        self.frequences_ = ...
        return self

    def transform(self, X):
        colonne = X.iloc[:, 0]
        # TODO 2 — remplace chaque modalité par sa fréquence apprise au fit ;
        # une modalité inconnue vaut 0.0  (indice : .map puis .fillna)
        encode = ...
        return encode.to_frame().to_numpy()

In [ ]:
# Cellule de test (donnée) — exécute-la telle quelle : 3 assertions.
_demo = pd.DataFrame({"clothing_id": ["A", "A", "A", "B"]})
_enc = FrequencyEncoder().fit(_demo)
assert _enc.transform(_demo).shape == (4, 1), "la sortie doit faire (n, 1)"
assert _enc.transform(_demo)[0, 0] == 0.75, "A apparaît 3 fois sur 4 -> 0.75"
assert _enc.transform(pd.DataFrame({"clothing_id": ["JAMAIS_VU"]}))[0, 0] == 0.0, \
    "une modalité inconnue doit valoir 0.0"
print("✅ FrequencyEncoder validé")

evalue("+ clothing_id en frequency encoding",
       branches_de_base() + [("cid", FrequencyEncoder(), ["clothing_id"])])

Une **seule** colonne au lieu de 1 100, et le f1_macro remonte au niveau de
la baseline. Rien gagné ? Patience — la fréquence ne dit encore rien du
**lien avec la cible**. C'est le job de l'encodage suivant.

## 🎯 [4] À toi — le target encoding (guidage moyen)

Le `TargetEncoder` de scikit-learn remplace chaque modalité par des
statistiques de **la cible** observées pour cette modalité — avec une
**validation croisée interne** pendant le `fit`.

> 🤔 Avant de coder, réponds pour toi : pourquoi cette CV interne est-elle
> vitale ? Que se passerait-il si chaque produit était encodé avec la
> moyenne de **sa propre** cible, vue telle quelle ? (Relis « fuite de
> cible » dans `fiche_preprocessing.pdf` si besoin.)

In [ ]:
from sklearn.preprocessing import TargetEncoder

# TODO 3 — construis la branche TargetEncoder pour clothing_id.
# Deux paramètres à choisir : target_type (notre cible a 3 classes) et
# random_state. Documentation : help(TargetEncoder) ou la doc en ligne.
branche_target = ("cid", ..., ["clothing_id"])

evalue("+ clothing_id en target encoding",
       branches_de_base() + [branche_target])

En **mixte**, tu viens de le mesurer : le texte écrase tout, `clothing_id`
bien encodé n'ajoute (presque) rien. Alors, du temps perdu ? Non — souviens-toi
de la demande de Léa : **les 845 avis sans texte**.

## 🎯 [5] À toi — le vrai test : sans le texte (guidage faible)

Pour un avis « note seule », le modèle n'a QUE le tabulaire. C'est là que ton
travail d'encodage se joue.

In [ ]:
# TODO 4 — construis quatre jeux de branches SANS la branche texte, et
# compare avec evalue() :
#   a) tabulaire seul (num + cat)
#   b) + class_name en OneHot
#   c) + class_name + clothing_id en frequency encoding
#   d) + class_name + clothing_id en target encoding
# Indice de départ : [b for b in branches_de_base() if b[0] != "txt"]

**Ta conclusion en 3 lignes** (remplace ce paragraphe) : que recommandes-tu à
Léa pour les avis sans texte, et pourquoi ?

---

### 🧭 Repères pour t'auto-vérifier (pas la solution)

- En **mixte** : le OneHot naïf doit finir **sous** la baseline ; frequency et
  target au même niveau qu'elle.
- **Sans texte** : une progression nette de a) vers d), avec des f1_macro
  entre ~0,31 et ~0,35. Le gain est modeste mais réel et **ordonné** — sur un
  jeu avec plus d'historique par produit, l'écart se creuse.
- ⚠️ Si ton target encoding sans texte dépasse ~0,45 : tu as une **fuite de
  cible** quelque part (encodage fait hors pipeline ? sur train+test ?).

## 🔎 Ce que tu viens de pratiquer

- **Haute cardinalité** : jamais de OneHot naïf au-delà de quelques dizaines
  de modalités — colonnes ×1 100 pour un score en baisse.
- **Frequency encoding** : 1 colonne, robuste aux modalités inconnues, à
  coder soi-même en 4 lignes dans un transformer scikit-learn.
- **Target encoding** : plus puissant car il regarde la cible — donc à
  toujours faire **dans** le pipeline, avec sa CV interne (fuite sinon).
- **La valeur d'une feature dépend du contexte** : noyée quand le texte est
  là, précieuse quand il manque.

Ce geste se transpose tel quel à toute variable à centaines/milliers de
modalités : code postal, référence machine, identifiant client…

## ⭐ Pour aller plus loin (optionnel)

- Ajoute `add_indicator=True` aux `SimpleImputer` : le fait-même qu'une
  valeur manque devient une colonne. Effet ?
- Regroupe les produits vus moins de 10 fois dans une modalité `"autre"`
  avant encodage : robustesse contre les produits rares ?
- Évalue le scénario d) uniquement sur les **vrais** avis sans texte du jeu
  de test (`X_test["review_text"] == ""`) : c'est le chiffre qui intéresse
  vraiment Léa.